# Modul 13: Klassische Modelle für Bild- und Signaldaten | Lösungen

## Überblick

Sie wandeln Bilder in Pixel-, Histogramm- und kantenorientierte Merkmale um und bewerten klassische Bildklassifikatoren. Anschließend zerlegen Sie Signale in gelabelte Fenster, extrahieren statistische und frequenzbasierte Merkmale und führen zeitlich korrekte Modellvergleiche durch.

**Zugehörige Vorlesungen**

- **Klassische Bildmodelle**
- **Signalmodelle**

## Lernziele

Nach der Bearbeitung können Sie:

- kleine Graustufenbilder als Pixel-, Histogramm- und HOG-ähnliche Merkmale vorbereiten.
- klassische Bildmodelle leakage-frei trainieren und Fehlklassifikationen sichtbar untersuchen.
- Signalfenster mit Zeit- und Frequenzmerkmalen beschreiben und zeitlich korrekt bewerten.

## Geprüfte Fähigkeiten

- Digits-Bildtensoren, Pixelmerkmale, regionale Histogramme und einfache HOG-Merkmale
- Pipelines, PCA, Konfusionsmatrix und visualisierte Modellfehler
- Fensterung, statistische Signalmerkmale, FFT, zeitlicher Split und Baselines

## Hinweise zur Bearbeitung

Dieses Lösungsnotebook enthält dieselben Aufgaben wie das Übungsnotebook sowie vollständige, ausführlich kommentierte Musterlösungen. Bearbeiten Sie nach Möglichkeit zuerst das Übungsnotebook und nutzen Sie dieses Dokument anschließend zur Kontrolle und Vertiefung.

- **Erwarteter Schwierigkeitsgrad:** fortgeschritten
- Verwenden Sie sprechende Variablennamen und prüfen Sie wichtige Zwischenformen und Wertebereiche.
- Verändern Sie die vorgegebenen Zufalls-Startwerte nur, wenn eine Aufgabe dies ausdrücklich verlangt.
- Interpretieren Sie Ergebnisse fachlich. Eine einzelne Kennzahl ist selten eine vollständige Begründung.
- Alle Aufgaben sind für die kostenlose Google-Colab-Umgebung ausgelegt. Die Datensätze und Modelle sind bewusst klein gehalten. Eine GPU ist nicht erforderlich, kann aber bei einzelnen Deep-Learning-Aufgaben die Laufzeit verkürzen.

## Einrichtung und gemeinsame Datenbasis

Die Setup-Zelle lädt den kleinen Digits-Datensatz und erzeugt ein synthetisches Sensorsignal mit zeitlich wechselnden Zuständen. Alle Berechnungen laufen auf der CPU und benötigen keine externen Dateien.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

ziffern = load_digits()
bilder = ziffern.images.astype(np.float64)
bild_labels = ziffern.target

bild_train, bild_test, label_train, label_test = train_test_split(
    bilder,
    bild_labels,
    test_size=0.25,
    stratify=bild_labels,
    random_state=RANDOM_SEED,
)

# Synthetisches Signal: drei Zustände mit unterschiedlichen dominanten Frequenzen.
abtastrate_hz = 50
segment_laenge = 250
zustandsfolge = [0, 1, 2, 0, 2, 1, 0, 1, 2, 0, 2, 1]
signal_abschnitte = []
signal_labels = []
for segment_index, zustand in enumerate(zustandsfolge):
    lokale_zeit = np.arange(segment_laenge) / abtastrate_hz
    frequenz = [1.5, 4.0, 8.0][zustand]
    amplitude = [1.0, 0.8, 0.55][zustand]
    abschnitt = amplitude * np.sin(2 * np.pi * frequenz * lokale_zeit)
    abschnitt += 0.15 * rng.normal(size=segment_laenge)
    abschnitt += 0.05 * segment_index
    signal_abschnitte.append(abschnitt)
    signal_labels.extend([zustand] * segment_laenge)

sensor_signal = np.concatenate(signal_abschnitte)
sensor_label = np.asarray(signal_labels)
zeit_s = np.arange(sensor_signal.size) / abtastrate_hz

print("Bilder:", bilder.shape)
print("Signalpunkte:", sensor_signal.shape)

### Aufgabe 1: Bilddaten prüfen und Pixelmerkmale vorbereiten

Visualisieren Sie je ein Trainingsbild der Klassen 0 bis 4. Wandeln Sie anschließend die 8-mal-8-Bilder in 64 Pixelmerkmale um und skalieren Sie die Pixelwerte in den Bereich 0 bis 1.

Trainieren Sie eine Pipeline aus StandardScaler und logistischer Regression. Berichten Sie Accuracy und Macro-F1 auf dem Testdatensatz.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

# ============================================================
# MUSTERLÖSUNG
# ============================================================

for klasse in range(5):
    beispiel_index = np.flatnonzero(label_train == klasse)[0]
    plt.figure()
    plt.imshow(bild_train[beispiel_index], cmap="gray")
    plt.title(f"Beispiel der Klasse {klasse}")
    plt.axis("off")
    plt.show()

# reshape(..., -1) erhält die Bildanzahl und faltet Höhe und Breite zu 64 Merkmalen zusammen.
X_bild_train_pixel = bild_train.reshape(len(bild_train), -1) / 16.0
X_bild_test_pixel = bild_test.reshape(len(bild_test), -1) / 16.0

pixel_modell = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=2500, random_state=RANDOM_SEED),
)
pixel_modell.fit(X_bild_train_pixel, label_train)
pixel_prognose = pixel_modell.predict(X_bild_test_pixel)

print("Pixel-Accuracy:", round(accuracy_score(label_test, pixel_prognose), 3))
print("Pixel-Macro-F1:", round(f1_score(label_test, pixel_prognose, average="macro"), 3))
print("Merkmalsform:", X_bild_train_pixel.shape)

> **Musterantwort und Interpretation**
>
> Die Bilder sollten dieselbe Größe, Kanalzahl und Orientierung besitzen und räumlich ähnlich ausgerichtet sein. Schon kleine Verschiebungen verändern viele Pixelspalten gleichzeitig. Pixelmerkmale sind deshalb eine nützliche Baseline, aber häufig weniger robust gegenüber Lage-, Beleuchtungs- oder Formvariationen als strukturorientierte Merkmale.

### Aufgabe 2: Regionale Histogramme als kompakte Bildmerkmale

Schreiben Sie eine Funktion `regionale_histogramme`, die jedes 8-mal-8-Bild in vier 4-mal-4-Quadranten teilt. Berechnen Sie je Quadrant ein Histogramm mit vier festen Bins im Wertebereich 0 bis 16 und normieren Sie es auf Summe 1.

Erzeugen Sie damit 16 Merkmale pro Bild, trainieren Sie dieselbe Modellart wie zuvor und vergleichen Sie Leistung und Merkmalsanzahl mit der Pixelbaseline.

In [ ]:
def regionale_histogramme(bild_stapel):
    # Erwartete Eingabeform: (Anzahl, 8, 8)
    # Erwartete Ausgabeform: (Anzahl, 16)
    pass

# ============================================================
# MUSTERLÖSUNG
# ============================================================

def regionale_histogramme(bild_stapel):
    """Erzeugt vier normierte Intensitätshistogramme mit je vier Bins."""
    alle_merkmale = []
    for bild in bild_stapel:
        quadranten = [
            bild[:4, :4],
            bild[:4, 4:],
            bild[4:, :4],
            bild[4:, 4:],
        ]
        bild_merkmale = []
        for quadrant in quadranten:
            histogramm, _ = np.histogram(quadrant, bins=4, range=(0, 16))
            # Jeder Quadrant besitzt 16 Pixel. Die Normierung macht Bilder vergleichbarer.
            histogramm = histogramm.astype(float) / histogramm.sum()
            bild_merkmale.extend(histogramm)
        alle_merkmale.append(bild_merkmale)
    return np.asarray(alle_merkmale)

X_bild_train_hist = regionale_histogramme(bild_train)
X_bild_test_hist = regionale_histogramme(bild_test)

hist_modell = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=2500, random_state=RANDOM_SEED),
)
hist_modell.fit(X_bild_train_hist, label_train)
hist_prognose = hist_modell.predict(X_bild_test_hist)

vergleich_hist = pd.DataFrame(
    {
        "Darstellung": ["Pixel", "Regionale Histogramme"],
        "Merkmale": [X_bild_train_pixel.shape[1], X_bild_train_hist.shape[1]],
        "Accuracy": [
            accuracy_score(label_test, pixel_prognose),
            accuracy_score(label_test, hist_prognose),
        ],
        "Macro_F1": [
            f1_score(label_test, pixel_prognose, average="macro"),
            f1_score(label_test, hist_prognose, average="macro"),
        ],
    }
)
display(vergleich_hist.round(3))

> **Musterantwort und Interpretation**
>
> Ein Histogramm zählt Intensitäten, merkt sich aber nicht die genaue Position jedes Pixels. Regionale Histogramme bewahren grobe Lageinformationen durch Quadranten, verlieren jedoch feine Kanten und lokale Formen. Sie sind kompakter und teilweise robuster, können aber verschiedene Ziffern mit ähnlicher Helligkeitsverteilung verwechseln.

### Aufgabe 3: Einfache HOG-Merkmale und PCA vergleichen

Implementieren Sie eine vereinfachte HOG-Funktion. Berechnen Sie horizontale und vertikale Gradienten mit `np.gradient`, daraus Betrag und Orientierung im Bereich 0 bis 180 Grad. Teilen Sie das Bild in vier 4-mal-4-Zellen und erstellen Sie je Zelle ein gewichtetes Orientierungshistogramm mit neun Bins.

Trainieren Sie ein Modell auf den 36 HOG-Merkmalen. Vergleichen Sie es außerdem mit einer Pipeline aus skalierten Pixelmerkmalen, PCA mit 20 Komponenten und logistischer Regression.

In [ ]:
from sklearn.decomposition import PCA

def einfache_hog_merkmale(bild_stapel, anzahl_bins=9):
    pass

# ============================================================
# MUSTERLÖSUNG
# ============================================================

def einfache_hog_merkmale(bild_stapel, anzahl_bins=9):
    """Berechnet pro 4-mal-4-Zelle ein gewichtetes Orientierungshistogramm."""
    merkmalslisten = []
    bin_kanten = np.linspace(0.0, 180.0, anzahl_bins + 1)

    for bild in bild_stapel:
        gradient_y, gradient_x = np.gradient(bild.astype(float))
        betrag = np.hypot(gradient_x, gradient_y)
        orientierung = (np.degrees(np.arctan2(gradient_y, gradient_x)) + 180.0) % 180.0

        bild_merkmale = []
        for zeilen_start in [0, 4]:
            for spalten_start in [0, 4]:
                winkel_zelle = orientierung[zeilen_start:zeilen_start + 4, spalten_start:spalten_start + 4]
                betrag_zelle = betrag[zeilen_start:zeilen_start + 4, spalten_start:spalten_start + 4]
                histogramm, _ = np.histogram(
                    winkel_zelle,
                    bins=bin_kanten,
                    weights=betrag_zelle,
                )
                # L2-Normierung reduziert den Einfluss der absoluten Kantenstärke.
                histogramm = histogramm / (np.linalg.norm(histogramm) + 1e-8)
                bild_merkmale.extend(histogramm)
        merkmalslisten.append(bild_merkmale)

    return np.asarray(merkmalslisten)

X_bild_train_hog = einfache_hog_merkmale(bild_train)
X_bild_test_hog = einfache_hog_merkmale(bild_test)

hog_modell = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=2500, random_state=RANDOM_SEED),
)
pca_modell = make_pipeline(
    StandardScaler(),
    PCA(n_components=20, random_state=RANDOM_SEED),
    LogisticRegression(max_iter=2500, random_state=RANDOM_SEED),
)

hog_modell.fit(X_bild_train_hog, label_train)
pca_modell.fit(X_bild_train_pixel, label_train)

hog_prognose = hog_modell.predict(X_bild_test_hog)
pca_prognose = pca_modell.predict(X_bild_test_pixel)

bild_vergleich = pd.DataFrame(
    {
        "Modell": ["Pixel", "HOG-ähnlich", "Pixel + PCA"],
        "Eingabemerkmale": [64, X_bild_train_hog.shape[1], 20],
        "Accuracy": [
            accuracy_score(label_test, pixel_prognose),
            accuracy_score(label_test, hog_prognose),
            accuracy_score(label_test, pca_prognose),
        ],
    }
)
display(bild_vergleich.round(3))

> **Musterantwort und Interpretation**
>
> Die Hauptkomponenten werden aus Mittelwerten, Varianzen und Korrelationen der Eingabedaten gelernt. Würden Testbilder dabei einbezogen, beeinflussten sie bereits die Merkmalsdarstellung des Modells. Eine Pipeline passt Standardisierung und PCA ausschließlich auf dem Trainingsanteil an und wendet die gelernte Transformation anschließend unverändert auf Testdaten an.

### Aufgabe 4: Fehlklassifikationen mit Konfusionsmatrix und Bildern analysieren

Verwenden Sie das leistungsstärkste der drei Bildmodelle nach Test-Accuracy. Erstellen Sie eine Konfusionsmatrix. Bestimmen Sie anschließend das häufigste Verwechslungspaar außerhalb der Hauptdiagonalen und visualisieren Sie bis zu sechs entsprechende falsch klassifizierte Testbilder.

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# ============================================================
# MUSTERLÖSUNG
# ============================================================

modell_kandidaten = {
    "Pixel": (pixel_modell, X_bild_test_pixel, pixel_prognose),
    "HOG-ähnlich": (hog_modell, X_bild_test_hog, hog_prognose),
    "Pixel + PCA": (pca_modell, X_bild_test_pixel, pca_prognose),
}

bester_bildname = max(
    modell_kandidaten,
    key=lambda name: accuracy_score(label_test, modell_kandidaten[name][2]),
)
bestes_bildmodell, _, beste_bildprognose = modell_kandidaten[bester_bildname]
print("Gewähltes Bildmodell:", bester_bildname)

matrix = confusion_matrix(label_test, beste_bildprognose)
ConfusionMatrixDisplay(matrix).plot()
plt.title("Konfusionsmatrix des gewählten Bildmodells")
plt.show()

# Die Hauptdiagonale enthält korrekte Vorhersagen und wird für die Fehlersuche ausgeblendet.
fehler_matrix = matrix.copy()
np.fill_diagonal(fehler_matrix, 0)
wahre_klasse, vorhergesagte_klasse = np.unravel_index(np.argmax(fehler_matrix), fehler_matrix.shape)
print(f"Häufigste Verwechslung: wahr {wahre_klasse}, vorhergesagt {vorhergesagte_klasse}")

passende_fehler = np.flatnonzero(
    (label_test == wahre_klasse) & (beste_bildprognose == vorhergesagte_klasse)
)[:6]
for index in passende_fehler:
    plt.figure()
    plt.imshow(bild_test[index], cmap="gray")
    plt.title(f"Wahr: {label_test[index]}, vorhergesagt: {beste_bildprognose[index]}")
    plt.axis("off")
    plt.show()

> **Musterantwort und Interpretation**
>
> Die Gesamtkennzahl zeigt nicht, welche Klassen systematisch verwechselt werden und welche Bildvariationen problematisch sind. Konkrete Fehlbilder können schlechte Ausrichtung, schwache Striche, Mehrdeutigkeit oder Datenqualitätsprobleme sichtbar machen. Solche Erkenntnisse helfen bei der Entscheidung, ob Daten, Merkmale oder Modellierung verbessert werden sollten.

### Aufgabe 5: Signale in Fenster zerlegen und Merkmale extrahieren

Zerlegen Sie `sensor_signal` ohne Überlappung in Fenster von 100 Messpunkten. Weisen Sie jedem Fenster das Mehrheitslabel seiner Messpunkte zu.

Berechnen Sie je Fenster mindestens diese Merkmale: Mittelwert, Standardabweichung, Minimum, Maximum, RMS, Peak-to-Peak, dominante positive Frequenz und spektralen Schwerpunkt. Speichern Sie alles in einem DataFrame und prüfen Sie Form, Fehlwerte und Klassenverteilung.

In [ ]:
def extrahiere_signalmerkmale(signal_fenster, sampling_rate):
    pass

# ============================================================
# MUSTERLÖSUNG
# ============================================================

def extrahiere_signalmerkmale(signal_fenster, sampling_rate):
    """Beschreibt ein reelles Signal durch einfache Zeit- und Frequenzmerkmale."""
    signal_fenster = np.asarray(signal_fenster, dtype=float)
    frequenzen = np.fft.rfftfreq(signal_fenster.size, d=1.0 / sampling_rate)
    spektrum = np.abs(np.fft.rfft(signal_fenster - signal_fenster.mean()))

    # Die Gleichanteilskomponente bei 0 Hz wird für die dominante Frequenz ignoriert.
    spektrum_ohne_dc = spektrum.copy()
    spektrum_ohne_dc[0] = 0.0
    dominante_frequenz = frequenzen[np.argmax(spektrum_ohne_dc)]
    spektraler_schwerpunkt = np.sum(frequenzen * spektrum) / (np.sum(spektrum) + 1e-12)

    return {
        "Mittelwert": signal_fenster.mean(),
        "Standardabweichung": signal_fenster.std(),
        "Minimum": signal_fenster.min(),
        "Maximum": signal_fenster.max(),
        "RMS": np.sqrt(np.mean(signal_fenster**2)),
        "Peak_to_Peak": np.ptp(signal_fenster),
        "Dominante_Frequenz_Hz": dominante_frequenz,
        "Spektraler_Schwerpunkt_Hz": spektraler_schwerpunkt,
    }

fensterlaenge = 100
merkmalszeilen = []
for start in range(0, len(sensor_signal) - fensterlaenge + 1, fensterlaenge):
    ende = start + fensterlaenge
    fenster = sensor_signal[start:ende]
    labels_im_fenster = sensor_label[start:ende]
    mehrheitslabel = np.bincount(labels_im_fenster).argmax()

    zeile = extrahiere_signalmerkmale(fenster, abtastrate_hz)
    zeile["Startindex"] = start
    zeile["Zeit_s"] = start / abtastrate_hz
    zeile["Ziel"] = mehrheitslabel
    merkmalszeilen.append(zeile)

signal_tabelle = pd.DataFrame(merkmalszeilen)
print("Merkmalstabelle:", signal_tabelle.shape)
print("Fehlwerte:", int(signal_tabelle.isna().sum().sum()))
print("Klassenverteilung:")
print(signal_tabelle["Ziel"].value_counts().sort_index())
display(signal_tabelle.head())

> **Musterantwort und Interpretation**
>
> Größere Fenster enthalten mehr Schwingungsperioden und ermöglichen meist stabilere sowie feinere Frequenzschätzungen. Gleichzeitig wird die zeitliche Auflösung schlechter, Übergänge werden vermischt und es entstehen weniger Trainingsbeispiele. Die Fensterlänge muss deshalb zum physikalischen Prozess und zur gewünschten Reaktionszeit passen.

### Aufgabe 6: Integrationsaufgabe: zeitlich korrektes Signalmodell

Sortieren Sie die Fenstertabelle nach `Startindex`. Verwenden Sie die ersten 70 Prozent der Fenster als Training und die letzten 30 Prozent als Test, ohne zufälliges Mischen.

Vergleichen Sie einen `DummyClassifier(strategy="most_frequent")` mit einer Pipeline aus StandardScaler und logistischer Regression. Berechnen Sie Accuracy und Macro-F1, zeichnen Sie wahre und vorhergesagte Zustände über der Testzeit und erläutern Sie, warum ein zufälliger Split hier problematisch wäre.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score

# ============================================================
# MUSTERLÖSUNG
# ============================================================

signal_tabelle = signal_tabelle.sort_values("Startindex").reset_index(drop=True)
merkmalsspalten = [
    spalte
    for spalte in signal_tabelle.columns
    if spalte not in ["Startindex", "Zeit_s", "Ziel"]
]

trenn_index = int(len(signal_tabelle) * 0.70)
X_signal_train = signal_tabelle.loc[: trenn_index - 1, merkmalsspalten]
y_signal_train = signal_tabelle.loc[: trenn_index - 1, "Ziel"]
X_signal_test = signal_tabelle.loc[trenn_index:, merkmalsspalten]
y_signal_test = signal_tabelle.loc[trenn_index:, "Ziel"]
test_zeit = signal_tabelle.loc[trenn_index:, "Zeit_s"]

signal_modelle = {
    "Mehrheitsbaseline": DummyClassifier(strategy="most_frequent"),
    "Skalierte LogReg": make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=2000, random_state=RANDOM_SEED),
    ),
}

signal_berichte = []
signal_prognosen = {}
for name, signal_modell in signal_modelle.items():
    signal_modell.fit(X_signal_train, y_signal_train)
    prognose = signal_modell.predict(X_signal_test)
    signal_prognosen[name] = prognose
    signal_berichte.append(
        {
            "Modell": name,
            "Accuracy": accuracy_score(y_signal_test, prognose),
            "Macro_F1": f1_score(y_signal_test, prognose, average="macro", zero_division=0),
        }
    )

display(pd.DataFrame(signal_berichte).round(3))

plt.step(test_zeit, y_signal_test, where="post", label="Wahrer Zustand")
plt.step(
    test_zeit,
    signal_prognosen["Skalierte LogReg"],
    where="post",
    label="Vorhersage",
)
plt.xlabel("Zeit in Sekunden")
plt.ylabel("Zustandsklasse")
plt.title("Zeitlich geordnete Testvorhersagen")
plt.legend()
plt.show()

> **Musterantwort und Interpretation**
>
> Benachbarte Signalabschnitte stammen oft aus demselben Zustand und teilen Drift, Rauschmuster oder Betriebsbedingungen. Bei überlappenden Fenstern können sogar viele identische Messpunkte in Training und Test landen. Ein zufälliger Split misst dann eher Wiedererkennung naher Zeitabschnitte als Verallgemeinerung auf eine zukünftige Periode. Ein zeitlicher Split bildet den vorgesehenen Einsatz realistischer ab.

## Abschlusskontrolle

Prüfen Sie vor dem Abschluss:

- Lassen sich alle Zellen in sinnvoller Reihenfolge ausführen?
- Sind Formen, Datentypen, Wertebereiche und Zufalls-Startwerte dokumentiert?
- Wurden Trainings-, Validierungs- und Testinformationen sauber getrennt?
- Sind Diagramme und Kennzahlen beschriftet und fachlich interpretiert?
- Können Sie erklären, warum die gewählten Methoden zur Aufgabenstellung passen?